# Project Forge — ComfyUI no Kaggle (GPU gratuito)

Este notebook transforma o **Kaggle** em um servidor ComfyUI gratuito com GPU.

**Vantagem sobre o Colab:** quota de **~30 horas de GPU por semana** (Colab cai em 12h).

---

## COMO USAR (passo a passo):
1. Acesse [kaggle.com/notebooks](https://www.kaggle.com/notebooks) (crie conta grátis)
2. Clique em **New Notebook** → escolha este arquivo ou copie o conteúdo
3. **Ative a Internet**: menu do notebook → `Settings` → marque **Internet**
4. **Ative GPU**: `Settings` → `Accelerator` → **GPU T4 x2** (ou T4 / P100)
5. Execute todas as células (`Run All`)
6. Quando a URL do túnel aparecer, o **Forge detecta automaticamente** (sem colar nada)

---

> 💡 **Kaggle**: sessão dura até 9h; quota de GPU renova toda semana.
> Quando a sessão expirar, execute tudo de novo.
>
> ⚠️ **NOTA**: O Kaggle precisa de **Internet habilitada** (Settings) e o modelo
> (~7GB) é baixado na primeira execução. Depois da 1ª vez, fica em cache.

In [ ]:
# @title 1. Verificar GPU
import torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f"GPU: {gpu.name}")
    print(f"VRAM: {round(getattr(gpu, 'total_memory', getattr(gpu, 'total_mem', 0)) / 1024**3, 1)} GB")
    print(f"CUDNN: {torch.backends.cudnn.version()}")
else:
    print("❌ GPU não detectada. Vá em Settings → Accelerator → GPU T4 x2")
    raise SystemExit()

In [ ]:
# @title 2. Instalar dependências do sistema
!apt-get update -qq -y > /dev/null 2>&1
!apt-get install -qq -y git wget unzip zip libgl1 libglib2.0-0 libsm6 libxext6 libxrender-dev libgomp1 > /dev/null 2>&1

# Instalar cloudflared (túnel)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

print("✅ Dependências instaladas")

In [ ]:
# @title 3. Baixar/Instalar ComfyUI
import os

WORKDIR = "/kaggle/working"
COMFY_DIR = f"{WORKDIR}/ComfyUI"

if not os.path.exists(COMFY_DIR):
    print("Baixando ComfyUI...")
    !git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git "{COMFY_DIR}" 2>&1 | tail -1
else:
    print("ComfyUI já existe")

print("Instalando dependências Python...")
!pip install -q -r "{COMFY_DIR}/requirements.txt" 2>&1 | tail -1

print("✅ ComfyUI pronto")
print(f"Caminho: {COMFY_DIR}")

In [ ]:
# @title 4. Baixar modelo profissional Pixel Art (SDXL)
MODEL_NAME = "pixelArtDiffusionXL_spriteShaper.safetensors"
MODEL_PATH = f"{COMFY_DIR}/models/checkpoints/{MODEL_NAME}"

if not os.path.exists(MODEL_PATH) or os.path.getsize(MODEL_PATH) < 1_000_000_000:
    print("Baixando Pixel Art Diffusion XL - Sprite Shaper (~6.9GB)...")
    urls = [
        "https://huggingface.co/AIWorksMD/pixelArtDiffusionXL/resolve/main/pixelArtDiffusionXL_spriteShaper.safetensors",
        "https://huggingface.co/nncyberpunk/SDXL1.0_PixelArtDiffusionXL_SpriteShaper/resolve/main/SDXL1.0_PixelArtDiffusionXL_SpriteShaper.safetensors"
    ]
    ok = False
    for url in urls:
        try:
            !wget -q --show-progress -O "{MODEL_PATH}" "{url}"
            if os.path.exists(MODEL_PATH) and os.path.getsize(MODEL_PATH) > 1_000_000_000:
                ok = True
                break
            else:
                print("Falhou, tentando outra fonte...")
        except Exception as e:
            print(f"Erro na fonte {url}: {e}")
    if not ok:
        print("Nao foi possivel baixar o modelo.")
        raise SystemExit()
    print("Modelo profissional baixado!")
else:
    print("Modelo ja existe")

print(f"\nModelo: {MODEL_NAME}")
print(f"Tamanho: {round(os.path.getsize(MODEL_PATH) / 1024**3, 1)} GB")


In [ ]:
# @title 5. Baixar modelos de upscale (RealESRGAN x2 e x4)
import os
UPSCALE_DIR = f"{COMFY_DIR}/models/upscale_models"
os.makedirs(UPSCALE_DIR, exist_ok=True)

for UPSCALE_NAME in ["RealESRGAN_x2plus.pth", "RealESRGAN_x4plus.pth"]:
    UPSCALE_PATH = f"{UPSCALE_DIR}/{UPSCALE_NAME}"
    if not os.path.exists(UPSCALE_PATH):
        version = "v0.2.1" if "x2" in UPSCALE_NAME else "v0.1.0"
        print(f"Baixando {UPSCALE_NAME}...")
        !wget -q --show-progress -O "{UPSCALE_PATH}" "https://github.com/xinntao/Real-ESRGAN/releases/download/{version}/{UPSCALE_NAME}"
        print(f"Baixado: {UPSCALE_NAME}")
    else:
        print(f"Ja existe: {UPSCALE_NAME}")


In [ ]:
# @title 6. Instalar workflow Studio Pro (auto-carga)
# Baixa o workflow profissional do Project Forge direto do GitHub.

import urllib.request

WORKFLOW_URLS = [
    "https://raw.githubusercontent.com/Arte3D-Project-Forge/project-forge/main/config/workflows/forge_studio_v1_workflow.json",
    "https://raw.githubusercontent.com/Arte3D-Project-Forge/project-forge/main/config/workflows/comfyui_sprite_upscale_workflow.json",
]

os.makedirs(f"{COMFY_DIR}/user/default/workflows", exist_ok=True)

for url in WORKFLOW_URLS:
    name = url.split("/")[-1]
    dest = f"{COMFY_DIR}/user/default/workflows/{name}"
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "project-forge"})
        data = urllib.request.urlopen(req, timeout=60).read()
        with open(dest, "wb") as f:
            f.write(data)
        print(f"[OK] workflow salvo: {name}")
    except Exception as e:
        print(f"[!] falha ao baixar {name}: {e}")

print("\nWorkflows Studio Pro instalados.")
print("O Forge usa estes JSONs automaticamente ao gerar sprites.")

In [ ]:
# @title 7. Iniciar servidor ComfyUI
import subprocess, sys

server_port = 8188

log_file = open('/kaggle/working/comfyui.log', 'w')

server = subprocess.Popen(
    [sys.executable, "main.py", f"--port={server_port}", "--listen=0.0.0.0"],
    cwd=COMFY_DIR,
    stdout=log_file,
    stderr=subprocess.STDOUT,
    text=True
)

print(f"✅ ComfyUI iniciado (PID: {server.pid})")
print("Aguardando servidor ficar pronto...")

import time as tmod
import urllib.request
for i in range(120):
    try:
        req = urllib.request.Request("http://127.0.0.1:8188/system_stats")
        urllib.request.urlopen(req, timeout=2)
        print(f"✅ Servidor pronto após {i+1}s")
        break
    except:
        tmod.sleep(1)
else:
    print("❌ Servidor não iniciou. Verifique os logs em /kaggle/working/comfyui.log")
    server.terminate()
    raise SystemExit()


In [ ]:
# @title 8. Tunel automatico + sincronizacao (deixe rodando)
import subprocess, threading, time, re, urllib.request, json, os, sys

SYNC_URL = "https://jsonblob.com/api/jsonBlob/019fb890-6c80-7896-88e3-14ac5bf3ca7c"

TUNNEL_URL = [None]
TUNNEL_STOPPERS = []

def find_url(line):
    for pat in [r'https://[a-z0-9-]+\.lhr\.life',
                r'https://[a-z0-9-]+\.serveo\.net',
                r'https://[a-z0-9-]+\.trycloudflare\.com',
                r'https://[a-z0-9-]+\.loca\.lt']:
        m = re.search(pat, line)
        if m and "api.trycloudflare.com" not in m.group(0):
            return m.group(0)
    return None

def publish(url):
    try:
        data = json.dumps({"tunnel": url}).encode("utf-8")
        req = urllib.request.Request(
            SYNC_URL, data=data,
            headers={"Content-Type": "application/json"}, method="PUT")
        urllib.request.urlopen(req, timeout=15)
        print(f"[sync] URL publicada para o Forge: {url}")
    except Exception as e:
        print(f"[sync] erro ao publicar: {e}")

def drain(proc):
    try:
        for _ in proc.stdout:
            pass
    except Exception:
        pass

def try_ssh(host, timeout=45):
    p = subprocess.Popen(
        ["ssh", "-o", "StrictHostKeyChecking=no",
         "-o", "ServerAliveInterval=30", "-o", "ServerAliveCountMax=3",
         "-R", "80:localhost:8188", host],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    url_holder = [None]
    def reader():
        try:
            for line in p.stdout:
                line = line.strip()
                if not line:
                    continue
                u = find_url(line)
                if u and not url_holder[0]:
                    url_holder[0] = u
                    print(f"[tunel] URL ssh: {u}")
        except Exception:
            pass
    t = threading.Thread(target=reader, daemon=True)
    t.start()
    start = time.time()
    while time.time() - start < timeout:
        if url_holder[0]:
            return url_holder[0], p
        if p.poll() is not None:
            break
        time.sleep(0.3)
    return url_holder[0], p

def try_cloudflared(timeout=30):
    bin_path = "/tmp/cloudflared"
    if not os.path.exists(bin_path) or os.path.getsize(bin_path) < 1000000:
        print("[tunel] baixando cloudflared...")
        os.system(
            "curl -fsSL -o /tmp/cloudflared https://github.com/cloudflare/cloudflared/releases/download/2024.10.0/cloudflared-linux-amd64")
        if not os.path.exists(bin_path) or os.path.getsize(bin_path) < 1000000:
            return None, None
        os.chmod(bin_path, 0o755)
    cmd = [bin_path, "tunnel", "--url", "http://127.0.0.1:8188",
           "--no-autoupdate", "--protocol", "http2"]
    p = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    url_holder = [None]
    def reader():
        try:
            for line in p.stdout:
                u = find_url(line.strip())
                if u and not url_holder[0]:
                    url_holder[0] = u
                    print(f"[tunel] URL cloudflared: {u}")
        except Exception:
            pass
    threading.Thread(target=reader, daemon=True).start()
    start = time.time()
    while time.time() - start < timeout:
        if url_holder[0]:
            return url_holder[0], p
        if p.poll() is not None:
            break
        time.sleep(0.3)
    return url_holder[0], p

def stop_old_tunnels():
    for ev in list(TUNNEL_STOPPERS):
        ev.set()
    TUNNEL_STOPPERS.clear()
    os.system("pkill -f cloudflared >/dev/null 2>&1 || true")
    os.system("pkill -f 'ssh.*localhost.run' >/dev/null 2>&1 || true")
    os.system("pkill -f 'ssh.*serveo.net' >/dev/null 2>&1 || true")
    print("[tunel] processos antigos encerrados")

def run_tunnel():
    stopper = threading.Event()
    TUNNEL_STOPPERS.append(stopper)
    ssh_hosts = ["nokey@localhost.run", "serveo.net"]
    idx = 0
    while not stopper.is_set():
        host = ssh_hosts[idx % len(ssh_hosts)]
        print(f"[tunel] tentando ssh {host} ...")
        url, proc = try_ssh(host)
        if not url:
            print("[tunel] ssh sem resposta. tentando cloudflared ...")
            url, proc = try_cloudflared()
        if not url:
            idx += 1
            try:
                if proc: proc.kill()
            except Exception:
                pass
            print("[tunel] nao conseguiu URL. tentando de novo em 8s...")
            time.sleep(8)
            continue
        TUNNEL_URL[0] = url
        print(f"[tunel] ATIVO: {url}")
        publish(url)
        threading.Thread(target=drain, args=(proc,), daemon=True).start()
        try:
            proc.wait()
        except Exception:
            pass
        try:
            proc.kill()
        except Exception:
            pass
        TUNNEL_URL[0] = None
        if not stopper.is_set():
            print("[tunel] caiu. reconectando em 5s...")
            time.sleep(5)

stop_old_tunnels()
threading.Thread(target=run_tunnel, daemon=True).start()
print("Tunel automatico ATIVO. Deixe o Kaggle aberto.")

In [ ]:
# @title 9. Status do tunel (diagnostico)
import urllib.request, time
try:
    u = TUNNEL_URL[0] if 'TUNNEL_URL' in globals() else None
except Exception:
    u = None
print("Ultima URL conhecida:", u)
if u:
    try:
        r = urllib.request.urlopen(urllib.request.Request(f"{u}/system_stats"), timeout=10)
        print(f"ComfyUI respondeu: HTTP {r.status} (tunel OK)")
    except Exception as e:
        print(f"Tunel sem resposta: {e}")


In [ ]:
# @title 10. Instrucoes finais
print("""
========================================
   PROJECT FORGE - KAGGLE CONFIGURADO
========================================
   O tunel automatico esta rodando e publicando
   a URL para o Forge. Nao precisa colar nada!

   O Forge detecta a URL nova sozinho (a cada 8s)
   e mostra CONECTADO quando o servidor responde.

   No Forge:
   1. Abra CONFIGURACOES
   2. Gerador ativo = comfyui
   3. Clique em TESTAR - deve mostrar CONECTADO
   4. Vai no Studio e gere seus sprites!

   ATENCAO:
   - Deixe o Kaggle aberto e rodando.
   - Sessao dura ~9h. Quota de GPU ~30h/semana.
   - Quando a sessao expirar, rode tudo de novo.
""")
